# 02b -- Fine-tuning TinyUSFM na syntetycznym datasecie AFAST (psy/koty)

**Projekt:** AI-asystowana nawigacja USG -- Vet Eye S.A. (BiznesAI 15, Akademia Leona Kozminskiego)

**Cel notebooka:** Wytrenowac view classifier na 4 klasach syntetycznego protokolu AFAST
(CC, DH, HR, SR) uzywajac pretrained backbone TinyUSFM (5.5M params, ViT-Tiny).

**Dwuetapowy trening (transfer learning):**
1. **Iteracja 1 (warm-up / linear probing):** zamrozony backbone, trenujemy TYLKO glowice klasyfikacyjna. Cel: glowica uczy sie rozsadnych wag w przestrzeni pretrained features.
2. **Iteracja 2 (partial unfreezing):** odmrazamy N ostatnich blokow transformera + glowica. Cel: backbone delikatnie dostosowuje sie do nowej domeny.

**Dlaczego dwa etapy?** Zweryfikowane empirycznie 15.05.2026: pominiecie warm-up'u (skok od razu do partial unfreezing z losowa glowica) powoduje catastrophic forgetting -- overfit w 2 epoki. Warm-up stabilizuje trening.

**Dataset:** [`koscielnamarta/synthetic-usg-afast-vet`](https://huggingface.co/datasets/koscielnamarta/synthetic-usg-afast-vet) -- 4000 syntetycznych obrazow USG (256x256 grayscale, zbalansowane 4 klasy).

**Backbone:** [TinyUSFM](https://github.com/MacDunno/TinyUSFM) -- ViT-Tiny pretrained na 2 mln obrazow USG.

## 0. Instalacja i importy

In [ ]:
# Instalacja pakietow (Colab ma wiekszosci, ale upewniamy sie o wersje)
# WAZNE: timm==0.5.4 -- TinyUSFM wymaga dokladnie tej wersji (nowsze lamia import)
!pip install -q --upgrade huggingface_hub
!pip install -q timm==0.5.4
!pip install -q gdown scikit-learn

In [ ]:
import os
import sys
import json
import time
import glob
import shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image, ImageFilter

# Sprawdzenie GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("UWAGA: brak GPU. Trening potrwa dluzej, ale zadziala.")

## 1. Konfiguracja eksperymentu

In [ ]:
# ====================================================================
# JEDYNE MIEJSCE GDZIE ZMIENIASZ PARAMETRY MIEDZY EKSPERYMENTAMI
# ====================================================================

# --- Identyfikacja HF ---
HF_USER = "koscielnamarta"
DATASET_REPO = f"{HF_USER}/synthetic-usg-afast-vet"
MODEL_REPO   = f"{HF_USER}/synthetic-usg-afast-vet-classifier"

# --- Parametry treningu ---
NUM_CLASSES = 4                  # CC, DH, HR, SR
IMG_SIZE    = 224                # TinyUSFM wymaga 224x224
BATCH_SIZE  = 32

# Iteracja 1 (warm-up / linear probing):
I1_EPOCHS = 5
I1_LR     = 1e-3                # wyzszy LR bo trenujemy tylko glowice

# Iteracja 2 (partial unfreezing):
I2_EPOCHS = 8
I2_LR     = 1e-4                # 10x nizszy -- delikatny fine-tuning backbone
NUM_UNFROZEN_BLOCKS = 2          # ile ostatnich blokow transformera odmrazamy
                                 # Ablation: zmien na 1 i porownaj wyniki

WEIGHT_DECAY = 1e-4

# --- Foldery wynikowe (sesyjne -- po treningu pushujemy na HF) ---
RUN_NAME = f"I2_unfreeze{NUM_UNFROZEN_BLOCKS}blocks"
RUN_DIR  = f"/content/run_02b_{RUN_NAME}"
CKPT_DIR = f"{RUN_DIR}/checkpoints"
OUT_DIR  = f"{RUN_DIR}/outputs"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print("=== Konfiguracja ===")
print(f"Dataset:   {DATASET_REPO}")
print(f"Model:     {MODEL_REPO}")
print(f"I1:        {I1_EPOCHS} epok, lr={I1_LR} (head only)")
print(f"I2:        {I2_EPOCHS} epok, lr={I2_LR} (+ {NUM_UNFROZEN_BLOCKS} blokow)")
print(f"Run dir:   {RUN_DIR}")

## 2. Pobranie datasetu z Hugging Face

In [ ]:
from huggingface_hub import snapshot_download

print(f"Pobieram dataset: {DATASET_REPO}")
DATA_ROOT = snapshot_download(repo_id=DATASET_REPO, repo_type="dataset")
print(f"  -> {DATA_ROOT}")

TRAIN_DIR = os.path.join(DATA_ROOT, "dataset", "train")
TEST_DIR  = os.path.join(DATA_ROOT, "dataset", "test")
LOCAL_DATA_DIR = os.path.join(DATA_ROOT, "dataset")

CLASSES = sorted([d for d in os.listdir(TRAIN_DIR)
                  if os.path.isdir(os.path.join(TRAIN_DIR, d))])
assert len(CLASSES) == NUM_CLASSES, f"Oczekiwano {NUM_CLASSES} klas, znaleziono {len(CLASSES)}"

print(f"\nKlasy ({len(CLASSES)}): {CLASSES}")
for split_name, split_dir in [("train", TRAIN_DIR), ("test", TEST_DIR)]:
    print(f"\n{split_name}/:")
    for c in CLASSES:
        n = len(os.listdir(os.path.join(split_dir, c)))
        print(f"  {c}: {n} obrazow")

In [ ]:
# Sanity check -- znajdz uszkodzone pliki
bad_files = []
total = 0
for split in ['train', 'test']:
    for cls in CLASSES:
        cls_dir = os.path.join(LOCAL_DATA_DIR, split, cls)
        for path in glob.glob(os.path.join(cls_dir, '*.png')):
            total += 1
            try:
                if os.path.getsize(path) == 0:
                    bad_files.append((path, 'zero-byte'))
                    continue
                Image.open(path).verify()
            except Exception as e:
                bad_files.append((path, str(e)[:60]))

print(f"Sprawdzono: {total} plikow")
print(f"Uszkodzonych: {len(bad_files)}")
if bad_files:
    for path, reason in bad_files[:10]:
        print(f"  {os.path.relpath(path, LOCAL_DATA_DIR)} -> {reason}")
else:
    print("Wszystkie pliki OK")

## 3. Pobranie modelu TinyUSFM

In [ ]:
TINYUSFM_DIR = "/content/TinyUSFM"
CKPT_FILE    = "/content/tinyusfm_pretrained.pth"

if not os.path.exists(TINYUSFM_DIR):
    !git clone https://github.com/MacDunno/TinyUSFM.git {TINYUSFM_DIR}
    print("Repo TinyUSFM sklonowane")
else:
    print(f"Repo juz istnieje: {TINYUSFM_DIR}")

if not os.path.exists(CKPT_FILE):
    import gdown
    gdown.download(id="1P_wDuU-StALLJEBaymioaYu52CScW-l-", output=CKPT_FILE, quiet=False)
    print(f"Checkpoint pobrany: {os.path.getsize(CKPT_FILE) / 1e6:.1f} MB")
else:
    print(f"Checkpoint juz istnieje: {os.path.getsize(CKPT_FILE) / 1e6:.1f} MB")

if TINYUSFM_DIR not in sys.path:
    sys.path.insert(0, TINYUSFM_DIR)

In [ ]:
from model.tinyusfm import TinyUSFM

model = TinyUSFM()
state_dict = torch.load(CKPT_FILE, map_location="cpu")
model.load_state_dict(state_dict, strict=False)

# Podmiana glowicy na NUM_CLASSES klas
original_head_features = model.head.in_features  # 192
model.head = nn.Linear(original_head_features, NUM_CLASSES)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: TinyUSFM (ViT-Tiny)")
print(f"Parametry lacznie: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"Glowica: Linear({original_head_features} -> {NUM_CLASSES})")
print(f"Bloki transformera: {len(model.blocks)} (indeksy 0..{len(model.blocks)-1})")

## 4. Data loading (ImageFolder + augmentacje)

In [ ]:
# Augmentacje (TYLKO train): lekkie, bo USG ma znaczenie anatomiczne gora/dol
# NIE uzywamy VerticalFlip ani agresywnych znieksztalcen
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = ImageFolder(TRAIN_DIR, transform=train_transform)
test_dataset  = ImageFolder(TEST_DIR,  transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}
print(f"Mapowanie klas: {idx_to_class}")
print(f"Train: {len(train_dataset)} obrazow ({len(train_loader)} batchy)")
print(f"Test:  {len(test_dataset)} obrazow ({len(test_loader)} batchy)")

## 5. Funkcje treningowe (reuzywane w I1 i I2)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)
        running_loss += loss.item() * imgs.size(0)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return {
        'loss': running_loss / total,
        'acc': correct / total,
        'preds': np.array(all_preds),
        'labels': np.array(all_labels),
        'probs': np.array(all_probs),
    }


def run_training(model, train_loader, test_loader, criterion, optimizer,
                 epochs, device, ckpt_path, iteration_name=""):
    best_val_acc = 0.0
    best_eval = None
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*60}")
    print(f"  {iteration_name}")
    print(f"  Epoki: {epochs} | LR: {optimizer.param_groups[0]['lr']}")
    print(f"  Trenowane: {trainable:,} z {total_params:,} ({100*trainable/total_params:.2f}%)")
    print(f"{'='*60}")

    t_start = time.time()
    for epoch in range(1, epochs + 1):
        t_ep = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_result = evaluate(model, test_loader, criterion, device)
        val_loss, val_acc = val_result['loss'], val_result['acc']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            best_eval = val_result
            torch.save(model.state_dict(), ckpt_path)

        dt = time.time() - t_ep
        flag = " * BEST" if is_best else ""
        print(f"  Ep {epoch:2d}/{epochs} | "
              f"train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | "
              f"val_acc={val_acc:.4f}{flag} | "
              f"{dt:.1f}s")

    total_time = time.time() - t_start
    print(f"\n  Czas: {total_time:.1f}s | Best val_acc: {best_val_acc:.4f}")
    print(f"  Checkpoint: {ckpt_path}")
    return best_val_acc, history, best_eval

## 6. Funkcje wizualizacji wynikow

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report


def plot_learning_curves(history_i1, history_i2, out_path, dod_line=None):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    n1 = len(history_i1['train_loss'])
    n2 = len(history_i2['train_loss'])
    x1 = list(range(1, n1 + 1))
    x2 = list(range(n1 + 1, n1 + n2 + 1))

    axes[0].plot(x1, history_i1['train_loss'], 'b-o', ms=3, label='I1 train')
    axes[0].plot(x2, history_i2['train_loss'], 'r-o', ms=3, label='I2 train')
    axes[0].axvline(n1 + 0.5, color='gray', ls='--', alpha=0.5, label='I1->I2')
    axes[0].set(xlabel='Epoka', ylabel='Train loss', title='Train loss')
    axes[0].legend(fontsize=8)

    axes[1].plot(x1, history_i1['val_loss'], 'b-o', ms=3, label='I1 val')
    axes[1].plot(x2, history_i2['val_loss'], 'r-o', ms=3, label='I2 val')
    axes[1].axvline(n1 + 0.5, color='gray', ls='--', alpha=0.5)
    axes[1].set(xlabel='Epoka', ylabel='Val loss', title='Val loss')
    axes[1].legend(fontsize=8)

    axes[2].plot(x1, history_i1['val_acc'], 'b-o', ms=3, label='I1 val_acc')
    axes[2].plot(x2, history_i2['val_acc'], 'r-o', ms=3, label='I2 val_acc')
    axes[2].axvline(n1 + 0.5, color='gray', ls='--', alpha=0.5)
    if dod_line:
        axes[2].axhline(dod_line, color='green', ls=':', label=f'DoD={dod_line}')
    axes[2].set(xlabel='Epoka', ylabel='Val accuracy', title='Val accuracy')
    axes[2].set_ylim(0, 1.05)
    axes[2].legend(fontsize=8)

    plt.suptitle('Krzywe uczenia: Iteracja 1 (warm-up) -> Iteracja 2 (fine-tune)')
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Zapisano: {out_path}")


def plot_confusion_matrix(labels, preds, class_names, out_path):
    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, data, title, fmt in [(axes[0], cm, 'Counts', 'd'),
                                  (axes[1], cm_norm, 'Row-normalized (recall)', '.2f')]:
        im = ax.imshow(data, cmap='Blues', vmin=0)
        ax.set(xticks=range(len(class_names)), yticks=range(len(class_names)))
        ax.set_xticklabels(class_names, rotation=45, ha='right')
        ax.set_yticklabels(class_names)
        ax.set(xlabel='Predicted', ylabel='True', title=title)
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                val = data[i, j]
                txt = format(val, fmt)
                color = 'white' if data[i, j] > data.max() * 0.6 else 'black'
                ax.text(j, i, txt, ha='center', va='center', color=color, fontsize=9)
    plt.suptitle('Confusion Matrix')
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Zapisano: {out_path}")


def save_classification_report(labels, preds, class_names, history_i1, history_i2, out_path):
    report = classification_report(labels, preds, target_names=class_names, output_dict=True)
    data = {'classification_report': report, 'history_i1': history_i1, 'history_i2': history_i2}
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(classification_report(labels, preds, target_names=class_names))
    print(f"Zapisano: {out_path}")

## 7. Iteracja 1 -- Linear probing (warm-up)

Zamrazamy **caly** backbone. Trenujemy **tylko** nowa glowice (`model.head`)
+ warstwe normalizacyjna (`model.fc_norm`).

Cel: glowica uczy sie rozsadnych wag w przestrzeni pretrained features -- bez ruszania backbone.

In [ ]:
# Zamrozenie CALEGO modelu
for p in model.parameters():
    p.requires_grad = False

# Odmrozenie TYLKO glowicy + LayerNorm
for p in model.head.parameters():
    p.requires_grad = True
for p in model.fc_norm.parameters():
    p.requires_grad = True

trainable_i1 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Iteracja 1 (linear probing):")
print(f"  Trenowane: {trainable_i1:,} wag ({100*trainable_i1/total_params:.2f}% modelu)")
print(f"  Backbone: ZAMROZONY ({len(model.blocks)} blokow)")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer_i1 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=I1_LR, weight_decay=WEIGHT_DECAY
)

best_val_acc_i1, history_i1, eval_i1 = run_training(
    model, train_loader, test_loader, criterion, optimizer_i1,
    epochs=I1_EPOCHS, device=device,
    ckpt_path=os.path.join(CKPT_DIR, "best_i1.pt"),
    iteration_name="ITERACJA 1 -- Linear probing (warm-up)"
)

## 8. Iteracja 2 -- Partial unfreezing (fine-tuning)

Odmrazamy **N ostatnich blokow** transformera (domyslnie 2).
LR jest 10x nizszy niz w I1 -- delikatny tuning pretrained wag.

**WAZNE:** Startujemy z wagami po I1 (glowica juz ustawiona) -- to stabilizuje trening.
Bez warm-up'u model overfit'uje w 2 epoki (zweryfikowane empirycznie).

In [ ]:
total_blocks = len(model.blocks)
unfrozen_indices = list(range(total_blocks - NUM_UNFROZEN_BLOCKS, total_blocks))

for idx in unfrozen_indices:
    for p in model.blocks[idx].parameters():
        p.requires_grad = True

trainable_i2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Iteracja 2 (partial unfreezing):")
print(f"  Odmrozone bloki: {unfrozen_indices} (z {total_blocks} lacznie)")
print(f"  Trenowane: {trainable_i2:,} wag ({100*trainable_i2/total_params:.2f}% modelu)")
print(f"  Przyrost vs I1: +{trainable_i2 - trainable_i1:,} wag")

In [ ]:
optimizer_i2 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=I2_LR, weight_decay=WEIGHT_DECAY
)

best_val_acc_i2, history_i2, eval_i2 = run_training(
    model, train_loader, test_loader, criterion, optimizer_i2,
    epochs=I2_EPOCHS, device=device,
    ckpt_path=os.path.join(CKPT_DIR, "best_i2.pt"),
    iteration_name=f"ITERACJA 2 -- Partial unfreezing ({NUM_UNFROZEN_BLOCKS} blokow)"
)

# Zaladuj best checkpoint I2 (peak val_acc)
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, "best_i2.pt"), map_location=device))
print(f"\nZaladowano best checkpoint I2: val_acc={best_val_acc_i2:.4f}")

## 9. Wyniki i wizualizacje

In [ ]:
print("=" * 60)
print("POROWNANIE ITERACJI")
print("=" * 60)
print(f"  I1 (linear probing):     val_acc = {best_val_acc_i1:.4f}  ({trainable_i1:>8,} wag, {100*trainable_i1/total_params:.2f}%)")
print(f"  I2 ({NUM_UNFROZEN_BLOCKS} bloki unfreeze):  val_acc = {best_val_acc_i2:.4f}  ({trainable_i2:>8,} wag, {100*trainable_i2/total_params:.2f}%)")
print(f"  Poprawa I1->I2: +{100*(best_val_acc_i2 - best_val_acc_i1):.2f} pp")
best_val_acc = best_val_acc_i2

In [ ]:
plot_learning_curves(history_i1, history_i2,
    out_path=os.path.join(OUT_DIR, "learning_curves.png"), dod_line=0.70)

In [ ]:
plot_confusion_matrix(eval_i2['labels'], eval_i2['preds'], CLASSES,
    out_path=os.path.join(OUT_DIR, "confusion_matrix.png"))

In [ ]:
save_classification_report(eval_i2['labels'], eval_i2['preds'], CLASSES,
    history_i1, history_i2, out_path=os.path.join(OUT_DIR, "classification_report.json"))

## 10. Stress test -- robustness na degradacje obrazu

Sprawdzamy jak model trzyma sie przy zakloceniach test-time.
Cel: pokazac ze 0.99+ accuracy to efekt prostego syntetycznego datasetu --
model NIE jest magiczny, ma realny gradient degradacji.

In [ ]:
def evaluate_stressed(model, base_dir, classes, transform_base, device,
                      noise_std=0.0, blur_radius=0.0, partial_mask=False):

    class StressedImageFolder(ImageFolder):
        def __init__(self, root, transform, ns, br, pm):
            super().__init__(root, transform=None)
            self.final_transform = transform
            self.ns, self.br, self.pm = ns, br, pm

        def __getitem__(self, index):
            path, target = self.samples[index]
            img = self.loader(path)
            if self.br > 0:
                img = img.filter(ImageFilter.GaussianBlur(radius=self.br))
            if self.ns > 0:
                arr = np.array(img).astype(np.float32)
                arr += np.random.randn(*arr.shape) * 255 * self.ns
                arr = np.clip(arr, 0, 255).astype(np.uint8)
                img = Image.fromarray(arr)
            if self.pm:
                arr = np.array(img).astype(np.float32)
                arr[arr.shape[0] // 2:, :] *= 0.15
                img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
            if self.final_transform:
                img = self.final_transform(img)
            return img, target

    ds = StressedImageFolder(base_dir, transform_base, noise_std, blur_radius, partial_mask)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [ ]:
stress_configs = [
    {"name": "Clean (baseline)",          "noise_std": 0.00, "blur_radius": 0.0, "partial_mask": False},
    {"name": "Mild noise (5%)",           "noise_std": 0.05, "blur_radius": 0.0, "partial_mask": False},
    {"name": "Moderate (10% + blur 1)",   "noise_std": 0.10, "blur_radius": 1.0, "partial_mask": False},
    {"name": "Heavy (20% + blur 2)",      "noise_std": 0.20, "blur_radius": 2.0, "partial_mask": False},
    {"name": "Partial image (dolna 1/2)", "noise_std": 0.00, "blur_radius": 0.0, "partial_mask": True},
]

print("Stress test -- accuracy przy degradacji obrazu:")
print("-" * 55)
stress_results = {}
for cfg in stress_configs:
    acc = evaluate_stressed(model, TEST_DIR, CLASSES, test_transform, device,
        noise_std=cfg["noise_std"], blur_radius=cfg["blur_radius"], partial_mask=cfg["partial_mask"])
    stress_results[cfg["name"]] = acc
    print(f"  {cfg['name']:35s}  val_acc = {acc:.4f}")

plt.figure(figsize=(9, 5))
names = list(stress_results.keys())
accs  = list(stress_results.values())
colors = ['#2ecc71' if a > 0.9 else '#f39c12' if a > 0.6 else '#e74c3c' for a in accs]
bars = plt.bar(range(len(names)), accs, color=colors)
plt.xticks(range(len(names)), names, rotation=25, ha='right', fontsize=9)
plt.ylim(0, 1.05)
plt.axhline(0.25, color='red', ls='--', alpha=0.5, label='Random (4 klasy)')
plt.ylabel('Val accuracy')
plt.title('Robustness -- accuracy vs image degradation')
plt.legend()
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{acc:.3f}', ha='center', fontsize=9)
plt.tight_layout()
stress_path = os.path.join(OUT_DIR, "stress_test.png")
plt.savefig(stress_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Zapisano: {stress_path}")

## 11. Upload na Hugging Face Model Hub

Pushujemy checkpoint + artefakty + model card. Po tym Drive nie jest potrzebny.

In [ ]:
from huggingface_hub import login, whoami
from huggingface_hub.utils import LocalTokenNotFoundError

try:
    user_info = whoami()
    print(f"Zalogowany jako: {user_info['name']}")
except (LocalTokenNotFoundError, Exception):
    print("Brak tokena HF -- loguje...")
    login()
    user_info = whoami()
    print(f"Zalogowany jako: {user_info['name']}")

In [ ]:
from huggingface_hub import create_repo, upload_folder, upload_file

create_repo(MODEL_REPO, repo_type="model", exist_ok=True, private=False)
print(f"Model repo: https://huggingface.co/{MODEL_REPO}")

MODEL_CARD = f"""---
license: mit
tags:
  - ultrasound
  - veterinary
  - AFAST
  - tinyusfm
  - image-classification
library_name: pytorch
pipeline_tag: image-classification
datasets:
  - {DATASET_REPO}
metrics:
  - accuracy
  - f1
base_model: MacDunno/TinyUSFM
---

# TinyUSFM fine-tuned for synthetic veterinary AFAST classification

Two-stage transfer learning on [{DATASET_REPO}](https://huggingface.co/datasets/{DATASET_REPO}).

| Stage | Params | Best val_acc | Epochs | LR |
|---|---|---|---|---|
| I1 linear probing | {trainable_i1:,} ({100*trainable_i1/total_params:.1f}%) | {best_val_acc_i1:.4f} | {I1_EPOCHS} | {I1_LR} |
| I2 unfreeze {NUM_UNFROZEN_BLOCKS} blocks | {trainable_i2:,} ({100*trainable_i2/total_params:.1f}%) | {best_val_acc_i2:.4f} | {I2_EPOCHS} | {I2_LR} |

Trained exclusively on synthetic data -- not clinically usable.
Source: [github.com/koscielnamarta/vet-eye-ai-usg-demo](https://github.com/koscielnamarta/vet-eye-ai-usg-demo)
"""

with open("/tmp/model_card.md", "w", encoding="utf-8") as f:
    f.write(MODEL_CARD)

upload_file(path_or_fileobj="/tmp/model_card.md", path_in_repo="README.md",
            repo_id=MODEL_REPO, repo_type="model", commit_message="Add model card")
print("Model card uploaded")

In [ ]:
print(f"Uploaduje: {RUN_DIR}")
for root, dirs, files in os.walk(RUN_DIR):
    for fn in files:
        fp = os.path.join(root, fn)
        rel = os.path.relpath(fp, RUN_DIR)
        sz = os.path.getsize(fp)
        print(f"  {rel} ({sz/1e3:.0f} kB)" if sz < 1e6 else f"  {rel} ({sz/1e6:.1f} MB)")

upload_folder(folder_path=RUN_DIR, repo_id=MODEL_REPO, repo_type="model",
    ignore_patterns=["*.tmp", "__pycache__/*", "best_i1.pt"],
    commit_message=f"Upload best checkpoint + artifacts (val_acc {best_val_acc:.4f})")
print(f"\nDone! https://huggingface.co/{MODEL_REPO}")

## 12. Pobranie artefaktow (do commit na GitHub)

In [ ]:
zip_path = f"/content/artefakty_02b_{RUN_NAME}.zip"
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', OUT_DIR)
print(f"ZIP: {zip_path} ({os.path.getsize(zip_path)/1e3:.0f} kB)")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Nie w Colabie -- pobierz recznie:", zip_path)